In [6]:
# Load Doc

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=PyPDFLoader("10 RAG Architectures.pdf")
documents=loader.load()

splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=splitter.split_documents(documents)
print(f"total chunks: {len(chunks)}")



total chunks: 20


## STEP 3 : Embed and store in FAISS

Now convert every chunk into a vector and stored in . this is the indexing step  -- you do it once, save it and reuse it.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

#load Embedding model
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed all Chunks and store in FAISS
vectorstore=FAISS.from_documents(chunks,embeddings)

# Save to disk so you dont Re-embed every time
vectorstore.save_local("faiss_index")

# Load later
vectorstore= FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7912.48it/s]


# What FAISS.from_documents() does internally
1. Takes each chunk's text
2. Passes it through embedding model → gets vector
3. Stores vector + original text in FAISS index
4. Returns a vectorstore you can query